# Demo MVP - Capazeta

Este notebook demuestra el flujo completo implementado para el MVP:

1. Carga del grafo multi-agente (`build_graph()`)
2. Enrutamiento automático hacia el dominio correspondiente
3. Consulta directa al RAG de MITRE ATT&CK sobre el índice FAISS
4. Visualización del diagrama del grafo

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from langchain_core.messages import HumanMessage
from capazeta.graph.builder import build_graph
from capazeta.rag import mitre

## 1. Construir y visualizar el grafo de agentes

In [ ]:
graph = build_graph()
print("Grafo multi-agente construido correctamente.")

## 2. Demostración de enrutamiento

Probamos con una consulta relacionada a **Threat Intelligence** para verificar que el router delega correctamente al subagente correspondiente.

In [ ]:
config = {"configurable": {"thread_id": "demo-mvp"}}
user_msg = HumanMessage(content="¿Cómo puedo mapear un login fallido sospechoso a técnicas MITRE ATT&CK?")
response = graph.invoke({"messages": [user_msg]}, config=config)

node = response.get('next_node')
print(f"\nAgente enrutado: {node}")
print("-" * 60)
print(response['messages'][-1].content)
print("-" * 60)

## 3. Consulta directa al RAG de MITRE ATT&CK (FAISS)

Esta prueba demuestra la funcionalidad de búsqueda semántica sobre el índice FAISS sin necesidad de ejecutar herramientas externas.

In [ ]:
try:
    resultados = mitre.search("brute force authentication", k=5)
    print("Técnicas MITRE ATT&CK recuperadas:\n")
    for idx, r in enumerate(resultados, 1):
        print(f"{idx}. [{r['id']}] {r['name']}")
        print(f"   Tácticas: {r['tactics']}")
        print(f"   URL: {r['url']}\n")
except Exception as e:
    print(f"Error: {e}")
    print("\nNota: Es necesario haber construido el índice con: uv run python -m capazeta.rag.build")

## 4. Consulta por ID específico

Ejemplo de enriquecimiento de técnicas utilizando el catálogo `techniques.json`.

In [ ]:
try:
    tech = mitre.get_technique("T1110")
    if tech:
        print(f"ID: {tech['id']}")
        print(f"Nombre: {tech['name']}")
        print(f"Tácticas: {', '.join(tech['tactics'])}")
        print(f"URL: {tech['url']}")
    else:
        print("Técnica no encontrada")
except Exception as e:
    print(f"Error: {e}")